# 3 · Hata ve serbest metin: kalibrasyon, ileri test, açıklama metni

Teknik raporun 8–9. bölümleri ([technical.tr.md](../reports/technical.tr.md)).

**Ne koşar, ne kayıttan gelir:**
- **Hafif betikler bu defterde koşar.** `script = run("<betik>")` betiğin [1]–[5] hücrelerini çalıştırır: veriyi ve
  kayıtlı model çıktılarını (OOF tahminleri, SHAP dosyası) okur, hesabı yapar; kaydet hücresi [6] çalışmaz.
  Ardından `check` sonucun `metrics/` kaydıyla aynı olup olmadığını gösterir. `script.res`, yüklenen tablolar
  (`script.listings` …) ve betiğin bütün fonksiyonları sonraki hücrelerde kullanılabilir.
- **Ağır betikler kayıttan okunur.** Model eğiten ya da çapraz doğrulama koşan betikler (`HEAVY`) için
  `recorded("<betik>", docs)` `metrics/` kaydını getirir.
- **Soru, veri, yöntem ve sızıntı notları** kartlardan (`analysis/cards/`), **figürler** derleyicilerin çizdiği
  `reports/figures/`'tan gelir.

Analiz kodu betiklerin kendisidir; ortak kod `analysis/lib` ve `builders/report_lib`'den gelir, burada kopyası yok. Bir
sonucu kalıcı olarak değiştirmek için betik değişir ve zincir koşulur; defter yeni kaydı okur. Defter dosya yazmaz.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis" / "lib").is_dir())
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "builders")]

import pandas as pd
from IPython.display import Markdown

from lib.records import card, check, flat, recorded, run, runs, show
from report_lib import metrics_view as MV

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 120)
SCRIPTS = ["08_residuals", "08_large_errors", "08_conformal_coverage", "08_forward_test", "10_free_text"]
docs = MV.load_metrics(ROOT / "metrics")
MV.check_consistency(docs)
runs(docs, SCRIPTS)

,yazıldı,veri sonu,DB,run_id
betik,,,,
08_residuals,2026-11-22T20:01:15+03:00,2026-06-27,444123b1f1a9,601741517e09
08_large_errors,2026-11-22T20:01:18+03:00,2026-06-27,444123b1f1a9,601741517e09
08_conformal_coverage,2026-11-23T18:24:34+03:00,2026-06-27,444123b1f1a9,601741517e09
08_forward_test,2026-11-23T18:25:07+03:00,2026-06-27,444123b1f1a9,NaN
10_free_text,2026-11-22T20:03:17+03:00,2026-06-27,444123b1f1a9,NaN


## 8. Kalibrasyon, artıklar ve zayıflık

### 08_residuals
[betik](../analysis/08_residuals.py) · [kart](../analysis/cards/08_residuals.json)

In [2]:
Markdown(card("08_residuals"))

**Soru:** OOF hataları nasıl dağılıyor; fiyat düzeyine göre yanlılık var mı?

**Veri:** load_clean() + 07_model_comparison'ın OOF tahminleri.

**Yöntem:** Hata bantları; tahmin edilen fiyatın çeyreklerine göre medyan hata, lira hatası payı ve sapma (gerçek fiyata göre gruplamak ortalamaya dönüş üretir); kalibrasyon eğimi; tahmin-gerçek ve artık figürlerinin noktaları.

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `none` — Tekilleştirme (ad_id başına son kayıt, ORDER BY ad_id) fold'lardan önce; aynı ad_id iki fold'a düşemez.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `present` — Kendisi model kurmaz; ama okuduğu OOF tahminleri 07'nin erken durdurmasını taşır (fold'un kendi doğrulama kısmında): hatalar hafifçe iyimser olabilir. Etkisi ölçülmedi.
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Son taramada yeni ilanlardaki hata 08_forward_test'te ayrıca ölçülüyor.

In [3]:
script = run("08_residuals")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["08_residuals"])

median % error by predicted quartile | tahmin çeyreğine göre medyan % hata: [('Q1', 6.95), ('Q2', 4.97), ('Q3', 4.2), ('Q4', 3.55)]


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [4]:
d, ed = m["domain"], m["error_drivers"]
show("tahmin ve gerçek (noktalar figürlerde)", flat({"pred_vs_true": d["pred_vs_true"], "residual_scatter": d["residual_scatter"]}))
show("tahmin fiyatı çeyreğine göre medyan |hata|", pd.DataFrame(d["quantile_error"], columns=["çeyrek", "medyan |hata| %"]))
show("tahmin fiyatı çeyreğine göre lira hatası",
     pd.DataFrame(ed["lira_quartile"], columns=["çeyrek", "ilan", "lira hatası payı %", "ortalama |hata| ₺", "medyan |hata| %"]))
show("tahmin fiyatı çeyreğine göre sapma (tahmin − gerçek)",
     pd.DataFrame(ed["pred_quartile"], columns=["çeyrek", "ilan", "ortalama sapma ₺", "medyan sapma ₺"]))
show("kalibrasyon (gerçek ~ a + b·tahmin)", flat(ed["calibration"]))
show("hata dağılımı: özet", flat(m["report"]))
show("hata bantları", pd.DataFrame(m["report"]["err_bands"], columns=["alt %", "üst %", "ilan", "pay %"]))
show("model başına (en az 5 ilan)", pd.DataFrame(d["residual_vs_n"], columns=["ilan", "medyan |hata| %", "medyan fiyat"]))

**tahmin ve gerçek (noktalar figürlerde)**

,değer
pred_vs_true.ideal_line,"345000.0, 6500000.0"
pred_vs_true.r2,0.9745
pred_vs_true.n,29988
pred_vs_true.note,"OOF tahmin (sızıntısız), TÜM noktalar. Frontend: yoğunluk/hexbin veya düşük-opacity ile render (30K nokta ham scatte..."
residual_scatter.mean_resid_pct,-0.47
residual_scatter.std_resid_pct,9.27
residual_scatter.n,29988
residual_scatter.note,"Artık% = (gerçek-tahmin)/gerçek, TÜM noktalar. Ortalama sıfıra yakın. Fiyata göre yanlılık tahmin edilen fiyata göre..."


**tahmin fiyatı çeyreğine göre medyan |hata|**

,çeyrek,medyan |hata| %
0,Q1,6.95
1,Q2,4.97
2,Q3,4.20
3,Q4,3.55


**tahmin fiyatı çeyreğine göre lira hatası**

,çeyrek,ilan,lira hatası payı %,ortalama |hata| ₺,medyan |hata| %
0,Q1,7497,17.6,77305.0,6.95
1,Q2,7497,19.9,87321.0,4.97
2,Q3,7497,23.6,103608.0,4.20
3,Q4,7497,38.9,170870.0,3.55


... (6 outputs left out)

**hata bantları**

,alt %,üst %,ilan,pay %
0,0,5.0,15775,52.604375
1,5,10.0,8221,27.414299
2,10,20.0,4784,15.953048
3,20,NaN,1208,4.028278


**model başına (en az 5 ilan)**

,ilan,medyan |hata| %,medyan fiyat
0,25,8.99,870000.0
1,15,4.17,835000.0
2,27,7.03,880000.0
3,30,5.54,967000.0
4,6,9.33,970000.0
...,...,...,...
433,6,11.31,1179500.0
434,15,3.69,2490000.0
435,10,6.42,2010000.0
436,22,7.48,4697500.0


![tr-08-pred-vs-true](../reports/figures/tr-08-pred-vs-true.png)

![tr-09-residual](../reports/figures/tr-09-residual.png)

![tr-28-residual-lira](../reports/figures/tr-28-residual-lira.png)

![tr-26-error-hist](../reports/figures/tr-26-error-hist.png)

![tr-10-quartile-error](../reports/figures/tr-10-quartile-error.png)

![tr-27-quartile-lira](../reports/figures/tr-27-quartile-lira.png)

### 08_large_errors
[betik](../analysis/08_large_errors.py) · [kart](../analysis/cards/08_large_errors.json)

In [5]:
Markdown(card("08_large_errors"))

**Soru:** Büyük hatalar (|OOF artık| > %20) nereden geliyor: emsal sayısı, model, yaş, segment, lira ölçeğinde en büyük hataların yeri, katalog uyuşmazlığı?

**Veri:** load_clean() (29.988 ilan) ve 07_model_comparison'ın OOF tahminleri (data/analysis/oof.parquet).

**Yöntem:** Büyük hata oranı emsal sayısına, modele, yaşa (her yaş ve her kesim), F/S segmente göre; en büyük 100 lira hatasının düşük/fazla, en pahalı çeyrek ve performans ailesi dağılımı; kendi modelinin medyanından sapan motor değerleri; notlarının dayandığı karşılaştırmalarla üç örnek (her koşuda hesaplanır).

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — Kırılımlar ilan özniteliklerinden; hata OOF tahminden.
- bütün veriye uydurma: `n/a`
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Son taramada yeni ilanlardaki hata 08_forward_test'te ayrıca ölçülüyor.

In [6]:
script = run("08_large_errors")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["08_large_errors"])

large errors | büyük hata 1208 (768 over / 440 under)


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [7]:
ed = m["error_drivers"]
show("büyük hata: özet", flat(ed))
show("emsal sayısına göre", pd.DataFrame(ed["by_model_year_n"]))
show("model başına ilan kovasına göre", pd.DataFrame(ed["per_model_buckets"]))
show("yaşa göre büyük hata oranı", pd.DataFrame(ed["age_sensitivity"], columns=["yaş", "ilan", "büyük hata %"]))
show("yaş kesimleri", pd.DataFrame(ed["age_cuts"], columns=["kesim t", "≥t ilan", "≥t büyük hata %", "<t büyük hata %", "oran"]))
show("örnek ilanlar", pd.DataFrame(ed["examples"]))
show("model başına (bütün modeller)", pd.DataFrame(ed["per_model_error"], columns=["ilan", "medyan |hata| %"]))

**büyük hata: özet**

,değer
threshold_pct,20.0
overall.n,29988
overall.n_big,1208
overall.n_over,768
overall.n_under,440
...,...
spec_outliers.blind_spot.model,307
spec_outliers.n,15
spec_outliers.pct,0.05
spec_outliers.median_error_pct,12.8


**emsal sayısına göre**

,bin,n,big_pct,over_pct,under_pct
0,1,610,19.02,10.66,8.36
1,2–4,1584,10.48,6.19,4.29
2,5–19,5732,4.73,3.02,1.71
3,20–99,16505,2.99,1.94,1.05
4,100+,5557,2.90,2.02,0.88


**model başına ilan kovasına göre**

,bin,lo,hi,n_models,n_listings,median_of_medians
0,1,1,1,126,126,8.89
1,2–4,2,4,181,506,8.47
2,5–19,5,19,188,1876,5.99
3,20–99,20,99,171,7946,4.71
4,100+,100,1000000000,79,19534,4.58


... (4 outputs left out)

**örnek ilanlar**

,key,name,year,km,price,pred,resid_pct,n_model,n_model_year,compare
0,"[640i, 2011, 5600000]",BMW 640i,2011,160000,5600000.0,3236953.0,42.2,3,2,"{'series_n': 51, 'others': [[2011, 3550000.0], [2012, 2485000.0]], 'peer': None}"
1,"[4.2 FSI Quattro R-tronic, 2008, 4690000]",Audi 4.2 FSI Quattro R-tronic (R8),2008,112550,4690000.0,2744815.0,41.5,1,1,"{'series_n': 1, 'others': [], 'peer': {'model': 'S5 4.2 FSI Quattro', 'n': 4, 'median': 2622500.0}}"
2,"[750i Long, 2007, 1190000]",BMW 750i Long,2007,271000,1190000.0,2831842.0,-138.0,2,1,"{'series_n': 244, 'others': [[2009, 5300000.0]], 'peer': {'model_prefix': '730d', 'same_year': True, 'n': 15, 'media..."


**model başına (bütün modeller)**

,ilan,medyan |hata| %
0,25,8.99
1,15,4.17
2,27,7.03
3,30,5.54
4,6,9.33
...,...,...
740,22,7.48
741,1,7.70
742,3,44.79
743,1,22.19


![tr-11-n-vs-error](../reports/figures/tr-11-n-vs-error.png)

### 08_conformal_coverage
[betik](../analysis/08_conformal_coverage.py) · [kart](../analysis/cards/08_conformal_coverage.json)

In [8]:
Markdown(card("08_conformal_coverage"))

**Soru:** %90 conformal aralık tahmin edilen fiyatın her bandında ve her araç türünde ilanların %90'ını kapsıyor mu; banda göre ayrı q (Mondrian) ve araca göre ölçek modeli ne değiştiriyor; isabet olasılığı güvenilir mi?

**Veri:** load_clean() + 07_model_comparison'ın OOF tahminleri.

**Yöntem:** Çapraz kalibrasyon: her kat yalnız öteki katların OOF log hatalarıyla kalibre edilir (sonlu örneklem yüzdeliği). Üç kol: tek q ve tahmin edilen fiyatın çeyrek bandı başına q (ön kayıt plans/08-mondrian-coverage); ölçek modeli (lib/conformal.py, karar notu 2026-11-23, ön kayıt yok): yarı genişlik k·s(x); s, OOF mutlak log hatasını ana modelin metin dışı kodlu girdilerinden, log tahminden ve emsal sayısından tahmin eden küçük bir LightGBM (emsalde artmayan, altta 0,02); puanlanan kat için s öteki dört kattan, k o dört katın içinde iç içe kurulan skorlardan. Kol başına kapsama, genişlik ve aralık skoru; ölçek kolunda bant, emsal kovası, girdi eksenleri ve görmediği eksenler başına kapsama; araç türü tablosu; son taramanın ilanlarında kapsama; fiyatlar %2/%5/%10 yüksek olsaydı kapsama (yalnız seviye, varsayım hesabı). İsabet olasılığı: aynı skorlardan P(|log hata| ≤ log 1,1); onda birlik dilimlerde öngörülen ve gerçekleşen, Brier skoru, sitede gösterilen değer başına gerçekleşen pay.

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `none` — Tekilleştirme (ad_id başına son kayıt, ORDER BY ad_id) fold'lardan önce; aynı ad_id iki fold'a düşemez.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `none` — Çapraz kalibreli: hiçbir ilanın kendi hatası kendi aralığını kalibre etmez (test_calibration_never_sees_the_scored_fold). Ölçek kolunda s puanlanan katı görmez (test_scale_never_sees_the_scored_fold) ve k o katın dışındaki skorlardan gelir (test_one_k_per_scored_fold). Bu yalnız s düzeyini kanıtlar: hataların kendisi ana modelin çapraz uydurmasından gelir ve emsal sayımı öteki dört katın tamamını kullanır; her şeyin öteki dört kattan yeniden kurulduğu sızıntısız kopyanın sayısı docs/decisions.md'de. Katlar rastgele olduğu için genel kapsama neredeyse tanım gereği ~%90; ileri kullanımdaki kapsama 08_forward_test'te. OOF hataları 07'nin erken durdurması yüzünden hafifçe iyimser olabilir. Ölçek modelinin ayarları ve ±%10 sonuçlar görüldükten sonra kondu, aranmadı.
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Son taramada yeni ilanlardaki hata 08_forward_test'te ayrıca ölçülüyor.

In [9]:
m = recorded("08_conformal_coverage", docs)

**Kayıttan okunuyor** (ağır betik: ölçek modeli için 26 küçük LightGBM eğitir (iç içe çapraz kalibrasyon)).

In [10]:
rp, dc = m["report"], m["domain"]["conformal"]
mo, sc, cf = rp["mondrian"], rp["scale"], rp["confidence"]
show("aralık: özet", flat({"conformal": dc, "report": {k: rp[k] for k in ("conformal_q", "conformal_all", "q_bounds")}}))
show("tahmin fiyatı bandına göre kapsama ve genişlik (tek q ve bant başına q)",
     pd.DataFrame(mo["by_band"], columns=mo["columns"]))
show("üç kol yan yana: tek q, bant başına q, ölçek modeli", pd.DataFrame(sc["arms"], columns=sc["arm_columns"]))
show("araç türüne göre: bugünkü tek q ve ölçek modeli", pd.DataFrame(sc["profiles"], columns=sc["profile_columns"]))
show("fiyatlar daha yüksek olsaydı kapsama (yalnız seviye; varsayım hesabı)",
     pd.DataFrame(sc["level_shift"], columns=sc["level_shift_columns"]))
show("isabet olasılığı: özet", flat({k: v for k, v in cf.items() if k != "by_shown"}))
show("isabet olasılığı: sitede gösterilen değere göre gerçekleşen pay",
     pd.DataFrame(cf["by_shown"], columns=["10'da", "ilan", "gerçekleşen %"]))

**aralık: özet**

,değer
conformal.coverage_target,90
conformal.note,Bantlar tahmin edilen fiyatın çeyrekleri; kapsama çapraz kalibreli (her kat öteki katlardan kalibre edilir). by_quan...
report.conformal_q,0.14123
report.conformal_all,89.99
report.q_bounds,"1154371.0315541304, 1532721.5318234982, 2256098.228932405"


**tahmin fiyatı bandına göre kapsama ve genişlik (tek q ve bant başına q)**

,band,n,coverage_global,coverage_band,width_pct_global,width_pct_band
0,Q1,7497,80.99,89.96,28.3,37.1
1,Q2,7497,90.98,90.04,28.3,27.2
2,Q3,7497,93.25,90.01,28.3,24.4
3,Q4,7497,94.72,90.04,28.3,21.4


**üç kol yan yana: tek q, bant başına q, ölçek modeli**

,arm,coverage,width_median,width_mean,interval_score,coverage_no_comparable
0,global,89.99,28.3,28.3,0.4211,70.74
1,band,90.01,25.9,27.5,0.4059,66.76
2,scale,90.09,24.7,27.2,0.3657,90.29


... (4 outputs left out)

**isabet olasılığı: özet**

,değer
mean_predicted,78.3
mean_observed,78.3
brier,0.1493
brier_constant,0.1701
skill_pct,12.2
p10,57.9
p50,80.8
p90,95.4
within_pct,10
shown_max,9


**isabet olasılığı: sitede gösterilen değere göre gerçekleşen pay**

,10'da,ilan,gerçekleşen %
0,1,7,14.3
1,2,106,30.2
2,3,280,40.4
3,4,868,48.8
4,5,2506,57.1
5,6,4637,64.4
6,7,6118,74.7
7,8,6983,84.8
8,9,8483,94.2


![tr-12-coverage](../reports/figures/tr-12-coverage.png)

![tr-31-hit-reliability](../reports/figures/tr-31-hit-reliability.png)

### 08_forward_test
[betik](../analysis/08_forward_test.py) · [kart](../analysis/cards/08_forward_test.json)

In [11]:
Markdown(card("08_forward_test"))

**Soru:** Son taramadan önceki ilanlarla eğitilen model, son taramada ilk kez görülen ilanları ne kadar iyi fiyatlıyor; %90 aralık (tek q, bant başına q, ölçek modeli) ve isabet olasılığı orada ne kadar tutuyor?

**Veri:** load_clean(all_snapshots=True): her taramanın her satırı. Eğitim: son taramadan önce görülen her ilan, son fiyatıyla. Test: son taramada ilk kez görülen ilanlar.

**Yöntem:** Manşet kurulum (cv.fold_matrices + cv.LGB_PARAMS, log1p fiyat). Ağaç sayısı eğitim kümesinde erken durdurmalı iç 5-fold CV'nin medyanı, sonra bütün eğitim kümesinde tek eğitim (servis edilen model gibi). MAPE ve test ilanlarını modele göre yeniden örnekleyen bootstrap'ten %95 aralık. Servis edilen aralık tarifi ileriye uygulanır: q eğitim kümesinin kendi OOF hatalarından, bant sınırları eğitim OOF tahminlerinden; tek q ve bant başına q ile kapsama genel, bant başına ve (model, yıl)'ı eğitimde olmayan ilanlarda. Ölçek modeli (lib/conformal.py) de ileriye uygulanır: s ve k eğitim kümesinin iç içe OOF skorlarından, servis edilen s bütün eğitim kümesiyle; eğitim ilanının emsali kendi iç katı sayılmadan, test ilanınınki bütün eğitim kümesinde sayılır. Kol başına genişlik, aralık skoru ve kaçırmanın yönü; ölçek genişliğinin beşte birlerinde kapsama; işaretli log hata; isabet olasılığının öngörülen ve gerçekleşen değeri. 09_backtest'in son biriken kurulumu (2026-11-22).

**Bölme:** `temporal`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Test yalnız eğitimde hiç görülmemiş ad_id'ler; ama farklı ad_id ile yeniden ilan edilen aynı araç eğitimde eski fiyatıyla bulunabilir. Payı küçük, etkisi ölçülmedi (backlog/ideas.md, 2026-10-12).
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `none` — Model, ağaç sayısı, q ve bant sınırları yalnız eğitim kümesinden; test taraması erken durdurmada da kullanılmaz (test_forward_coverage_calibrates_on_training_only). Ölçek kolu test fiyatını hiç almaz; k yalnız eğitim ilanlarından gelir (test_forward_scale_calibrates_on_training_only).
- zamansal: `none` — Eğitim son taramadan önce, test son taramada: zaman sırası korunur.

In [12]:
m = recorded("08_forward_test", docs)

**Kayıttan okunuyor** (ağır betik: son taramadan önceki ilanlarla model eğitimi (iç 5-fold CV ile)).

In [13]:
ft = m["methodology"]["forward_test"]
fc = ft["coverage"]
show("ileri test: son taramadan önceki ilanlarla eğit, son taramanın yeni ilanlarında ölç",
     flat({k: v for k, v in ft.items() if k != "coverage"}))
show("servis edilen aralık tarifi ileriye: kapsama", flat({k: v for k, v in fc.items() if k != "bands"}))
show("tahmin fiyatı bandına göre kapsama", pd.DataFrame(fc["bands"], columns=["bant", "ilan", "tek pay", "banda göre"]))
show("üç kol yan yana (yeni ilanlar)", pd.DataFrame(fc["arms"], columns=fc["arm_columns"]))
show("ölçek genişliğinin beşte birlerinde kapsama (en dar önce)",
     pd.DataFrame(fc["by_width_fifth"], columns=["beşte bir", "ilan", "tek pay", "banda göre", "ölçek modeli"]))
show("isabet olasılığı (yeni ilanlar): sitede gösterilen değere göre gerçekleşen pay",
     pd.DataFrame(ft["confidence"]["by_shown"], columns=["10'da", "ilan", "gerçekleşen %"]))

**ileri test: son taramadan önceki ilanlarla eğit, son taramanın yeni ilanlarında ölç**

,değer
train_until,2026-03-21
test,2026-06-27
n_train,21099
n,8889
MAPE,6.78
ci_lo,6.42
ci_hi,7.17
trees,211
signed_error.mean,2.46
signed_error.median,2.02


**servis edilen aralık tarifi ileriye: kapsama**

,değer
q,0.1405
q_band,"0.1834, 0.1349, 0.1221, 0.106"
bounds,"1144837.0, 1519016.0, 2243494.0"
n,8889
coverage_global,87.4
coverage_band,87.99
no_comparable.n,297
no_comparable.coverage_global,68.35
no_comparable.coverage_band,65.32
scale.k,2.0997


**tahmin fiyatı bandına göre kapsama**

,bant,ilan,tek pay,banda göre
0,Q1,2215,77.07,88.85
1,Q2,2248,88.57,87.32
2,Q3,2213,90.74,87.44
3,Q4,2213,93.22,88.34


... (2 outputs left out)

**ölçek genişliğinin beşte birlerinde kapsama (en dar önce)**

,beşte bir,ilan,tek pay,banda göre,ölçek modeli
0,1,1778,97.98,95.33,86.84
1,2,1778,95.28,92.91,86.05
2,3,1777,90.88,89.59,86.38
3,4,1778,82.28,85.15,85.26
4,5,1778,70.58,76.94,89.60


**isabet olasılığı (yeni ilanlar): sitede gösterilen değere göre gerçekleşen pay**

,10'da,ilan,gerçekleşen %
0,2,36,36.1
1,3,60,35.0
2,4,239,46.0
3,5,789,55.0
4,6,1139,59.5
5,7,1692,68.8
6,8,2454,81.1
7,9,2480,91.5


## 9. Serbest metin: ölçüldü, dahil edilmedi

### 10_free_text
[betik](../analysis/10_free_text.py) · [kart](../analysis/cards/10_free_text.json)

In [14]:
Markdown(card("10_free_text"))

**Soru:** Satıcı açıklaması modele eklenince OOF doğruluğu ne kadar değişir? (ön kayıt: plans/10-text-contribution)

**Veri:** load_clean() (modelin ilanları) ve açıklamaları (description_text); bu koşunun model OOF'u (07_model_comparison, cv.load_oof).

**Yöntem:** Taban kol = modelin OOF'u. Metin kolu = aynı 5 fold, öznitelikler ve LightGBM ayarları + açıklamanın kelime TF-IDF'i (1–2 gram, min_df 5) ve SVD 50, ikisi de eğitim fold'unda kurulur. İki kolun log R², MAPE ve MAE'si ve farkları; H1: ΔR² < 0.005 ve MAPE iyileşmesi < 0,2 puan.

**Bölme:** `kfold_oof`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Modelle aynı fold'lar: farklı ad_id ile yeniden ilan edilen aynı araç fold'lar arasına düşebilir (01_dedup_leakage %0,46–%4,26); iki kolu eşit etkiler.
- hedef öznitelikte: `none` — Öznitelikler modelinkiler artı açıklama metni; fiyattan türeyen alan yok. Açıklama fiyatı yazabilir; ölçülen fark bunu da içerir.
- bütün veriye uydurma: `present` — TF-IDF ve SVD yalnız eğitim fold'unda kurulur (test_text_features_fit_on_train_only). Erken durdurma modeldeki gibi fold'un doğrulama kısmında; iki kolu eşit etkiler.
- zamansal: `present` — KFold dönemleri karıştırır, modeldeki gibi; iki kol aynı bölmeyi kullanır.

In [15]:
m = recorded("10_free_text", docs)

**Kayıttan okunuyor** (ağır betik: açıklama metniyle 5-fold çapraz doğrulama).

In [16]:
ta = m["report"]["text_ablation"]
show("açıklama metni eklenince", flat(ta))

**açıklama metni eklenince**

,değer
r2_log_base,0.9699
r2_log_text,0.9707
delta_r2,0.0008
mape_base,6.49
mape_text,6.41
mae_base,109776
mae_text,108770
unexplained_share_pct,2.6
n,29988
folds,5
